# Book Recommendation System: Exploratory Data Analysis

This notebook documents the analytical side of the project. It covers dataset profiling, missing-value analysis, duplicate inspection, distribution analysis, and business-oriented observations that help justify the recommendation pipeline.

## 1. Setup

We import the project helpers so the notebook uses the exact same loading and cleaning logic as the application and training pipeline. This keeps the analysis consistent with the deployed recommendation system.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from wordcloud import WordCloud

from src.data_preprocessing import (
    build_modeling_frame,
    clean_datasets,
    dataset_summary,
    load_datasets,
)
from src.feature_engineering import build_popular_books, build_user_book_matrix

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='terrain')
plt.rcParams['figure.figsize'] = (12, 5)

## 2. Load Raw and Cleaned Data

The project uses three files: books, users, and ratings. We load the raw bundle first, then apply the shared cleaning pipeline so we can compare pre-cleaning and post-cleaning characteristics.

In [ ]:
raw_bundle = load_datasets()
clean_bundle = clean_datasets(raw_bundle)
interactions = build_modeling_frame(clean_bundle)

raw_books, raw_users, raw_ratings = raw_bundle.books, raw_bundle.users, raw_bundle.ratings
books, users, ratings = clean_bundle.books, clean_bundle.users, clean_bundle.ratings

print('Raw books shape:', raw_books.shape)
print('Raw users shape:', raw_users.shape)
print('Raw ratings shape:', raw_ratings.shape)
print('Filtered modeling frame:', interactions.shape)

### Observation

The modeling frame is substantially smaller than the raw ratings table because the recommender intentionally keeps only explicit ratings and then filters out inactive users and rarely rated books. This is useful because collaborative filtering works better when the retained interactions contain meaningful signal.

## 3. Dataset Overview

A quick structural summary helps us validate column names, data types, duplicates, and missing values before deeper analysis.

In [ ]:
overview = pd.DataFrame([
    dataset_summary(books, 'books'),
    dataset_summary(users, 'users'),
    dataset_summary(ratings, 'ratings'),
])
overview[['dataset', 'shape', 'duplicate_rows']]

In [ ]:
books.head(3)

In [ ]:
users.head(3)

In [ ]:
ratings.head(3)

### Observation

The books table contains mostly categorical metadata, the users table mixes categorical and numeric user profile information, and the ratings table acts as the interaction backbone of the recommender. This table layout is a classic setup for collaborative filtering.

## 4. Missing Value Analysis

Missing values affect data quality, visualization accuracy, and downstream feature engineering. We inspect them both numerically and visually.

In [ ]:
missing_summary = pd.DataFrame({
    'books_missing': books.isna().sum(),
    'users_missing': users.reindex(columns=['User-ID', 'Location', 'Age']).isna().sum(),
}).fillna(0).astype(int)
missing_summary

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(missing_summary.T, annot=True, fmt='d', cmap='YlOrBr')
plt.title('Missing Values Heatmap')
plt.show()

### Observation

The most noticeable missingness is in the user `Age` field, which is common in public recommendation datasets. The recommendation engine is not heavily dependent on age, so cleaning this field without forcing imputation reduces noise and avoids introducing artificial user patterns.

## 5. Duplicate Analysis

Duplicate rows can inflate popularity statistics and distort similarity computations. We confirm the cleaned data is safe to use.

In [ ]:
duplicate_counts = pd.Series({
    'books_duplicates': books.duplicated().sum(),
    'users_duplicates': users.duplicated().sum(),
    'ratings_duplicates': ratings.duplicated().sum(),
})
duplicate_counts

### Observation

The cleaned tables are duplicate-safe, which is important because repeated interactions would otherwise overemphasize some books and produce misleading nearest-neighbor relationships.

## 6. Data Types

Data type inspection verifies that IDs, ratings, and publication years are usable for numeric analysis.

In [ ]:
dtype_report = pd.DataFrame({
    'books': books.dtypes.astype(str),
    'users': users.reindex(columns=['User-ID', 'Location', 'Age']).dtypes.astype(str),
    'ratings': ratings.reindex(columns=['User-ID', 'ISBN', 'Book-Rating']).dtypes.astype(str),
})
dtype_report

### Observation

The cleaned schema gives numeric types to ratings and publication years while preserving identifiers and descriptive metadata as text. This is exactly what we want for both analytics and pivot-table generation.

## 7. Distribution of Ratings

Understanding rating frequencies is essential because collaborative filtering is sensitive to interaction sparsity and score concentration.

In [ ]:
rating_distribution = interactions['Book-Rating'].value_counts().sort_index()
rating_distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(x=rating_distribution.index, y=rating_distribution.values, ax=axes[0], color='#a7682a')
axes[0].set_title('Count of Explicit Ratings')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

sns.histplot(interactions['Book-Rating'], bins=10, kde=True, ax=axes[1], color='#447c69')
axes[1].set_title('Rating Histogram')
axes[1].set_xlabel('Rating')
plt.tight_layout()
plt.show()

### Observation

The filtered dataset is skewed toward higher explicit ratings, which is common in book communities where readers usually rate books they feel strongly about. This means cosine similarity will mostly compare positive preference signals rather than neutral behavior.

## 8. Publication Year Distribution

Publication year helps us understand catalog recency and spot whether the dataset is dominated by a specific era of publishing.

In [ ]:
plt.figure(figsize=(12, 4))
sns.histplot(books['Year-Of-Publication'], bins=40, color='#6f4e37')
plt.title('Distribution of Publication Years')
plt.xlabel('Publication Year')
plt.ylabel('Count')
plt.show()

In [ ]:
books_per_year = books['Year-Of-Publication'].value_counts().sort_index()
books_per_year.tail(20).plot(kind='line', color='#8c5e34', title='Books Published Per Year (Recent Window)')
plt.xlabel('Year')
plt.ylabel('Books')
plt.show()

### Observation

The dataset is concentrated in modern publication periods, which makes sense for digitally cataloged ratings. This helps explain why many recommendations cluster around contemporary popular fiction.

## 9. Most Popular Books

This view highlights the titles with the highest interaction volume. Popularity matters because these books are more likely to provide stable similarity neighborhoods.

In [ ]:
top_books = interactions.groupby('Book-Title')['Book-Rating'].count().sort_values(ascending=False).head(15)
top_books

In [ ]:
top_books.sort_values().plot(kind='barh', color='#c28f2c', title='Top 15 Most Rated Books')
plt.xlabel('Number of Ratings')
plt.ylabel('Book Title')
plt.show()

### Observation

A relatively small set of titles captures a large share of user attention. This is useful operationally because these anchor books often produce the most reliable recommendations and are excellent candidates for the app homepage.

## 10. Top Authors

Author-level activity reveals whose works dominate the interaction space and whether recommendation coverage is concentrated around a few famous names.

In [ ]:
top_authors = interactions.groupby('Book-Author')['Book-Rating'].count().sort_values(ascending=False).head(15)
top_authors

In [ ]:
top_authors.sort_values().plot(kind='barh', color='#8d6a9f', title='Top 15 Authors by Rating Volume')
plt.xlabel('Number of Ratings')
plt.ylabel('Author')
plt.show()

### Observation

Popular authors naturally appear often because multiple titles from the same author accumulate ratings. This can create strong recommendation clusters, especially for franchise or series-based reading behavior.

## 11. Publisher Analysis

Publisher concentration gives a sense of catalog diversity and can reveal whether the dataset is dominated by a handful of large publishing houses.

In [ ]:
top_publishers = books['Publisher'].value_counts().head(15)
top_publishers

In [ ]:
top_publishers.sort_values().plot(kind='barh', color='#4e7d6f', title='Top 15 Publishers in the Catalog')
plt.xlabel('Number of Books')
plt.ylabel('Publisher')
plt.show()

### Observation

Publisher volume is uneven, which is typical in large real-world catalogs. This does not directly drive the model, but it is useful context for understanding why some books and authors are overrepresented in the recommendation space.

## 12. Popularity-Based Recommendation Baseline

Before using similarity-based recommendations, we inspect the popularity fallback catalog. This is important for homepage browsing and cold-start scenarios.

In [ ]:
popular_books = build_popular_books(interactions)
popular_books.head(10)

### Observation

The popularity baseline is useful even in a collaborative filtering project because it gives a dependable fallback when a user has not selected a book yet. It also keeps the homepage immediately engaging.

## 13. Ratings Per Book and Rating Spread

These distributions help us inspect sparsity and detect whether books receive a balanced range of ratings or mostly high scores.

In [ ]:
book_rating_counts = interactions.groupby('Book-Title')['Book-Rating'].count()
book_average_ratings = interactions.groupby('Book-Title')['Book-Rating'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(book_rating_counts, bins=30, ax=axes[0], color='#9c6644')
axes[0].set_title('Number of Ratings per Book')
axes[0].set_xlabel('Ratings Count')

sns.boxplot(x=book_average_ratings, ax=axes[1], color='#7b9e87')
axes[1].set_title('Box Plot of Average Book Ratings')
axes[1].set_xlabel('Average Rating')
plt.tight_layout()
plt.show()

### Observation

Most books still sit on the sparse side even after filtering, which reinforces why thresholding is necessary. The box plot also shows that average ratings tend to remain in the upper range, again reflecting a positivity bias in explicit reviews.

## 14. Correlation Analysis

Correlation is most useful on numeric summaries rather than raw identifiers. We build a compact numeric view of book-level aggregates.

In [ ]:
book_level = interactions.groupby('Book-Title').agg(
    average_rating=('Book-Rating', 'mean'),
    ratings_count=('Book-Rating', 'count'),
    publication_year=('Year-Of-Publication', 'median')
).reset_index()

book_level[['average_rating', 'ratings_count', 'publication_year']].corr()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(
    book_level[['average_rating', 'ratings_count', 'publication_year']].corr(),
    annot=True,
    cmap='YlGnBu',
    fmt='.2f'
)
plt.title('Correlation Heatmap of Book-Level Features')
plt.show()

### Observation

There is usually only a modest relationship between popularity and rating quality in these datasets. That is a strong argument for using collaborative filtering rather than relying only on popularity or average score.

## 15. Pair Plot of Numeric Book-Level Features

Pair plots are best used on a manageable sample of numeric features so the relationships remain readable.

In [ ]:
pairplot_sample = book_level[['average_rating', 'ratings_count', 'publication_year']].sample(
    n=min(300, len(book_level)),
    random_state=42,
)
sns.pairplot(pairplot_sample, corner=True, diag_kind='hist')
plt.show()

### Observation

The pair plot helps confirm that the dataset does not separate cleanly into simple linear trends. That is exactly the kind of setting where neighborhood-based recommendation methods remain attractive because they rely on shared behavior instead of a single handcrafted rule.

## 16. Word Cloud of Book Titles

A title word cloud is not a modeling feature here, but it provides a fast qualitative sense of the catalog vocabulary.

In [ ]:
title_text = ' '.join(books['Book-Title'].dropna().astype(str).tolist()[:5000])
wordcloud = WordCloud(width=1200, height=500, background_color='white', colormap='copper').generate(title_text)

plt.figure(figsize=(14, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Book Titles')
plt.show()

### Observation

The title vocabulary emphasizes fiction-heavy catalog patterns, which aligns with the popularity of series, thrillers, and mainstream contemporary works in the recommendation results.

## 17. User-Book Matrix Preview

The pivot table is the core feature representation used by the recommender. Each row is a book, each column is a user, and each value is an explicit rating.

In [ ]:
pivot_table = build_user_book_matrix(interactions)
print('Pivot table shape:', pivot_table.shape)
pivot_table.iloc[:5, :10]

### Observation

The pivot table is intentionally sparse, which is normal for recommendation problems. Cosine similarity is a strong fit here because it compares the direction of user preference vectors rather than raw rating magnitude alone.

## 18. Final Conclusion

The exploratory analysis supports the design decisions used in the project:

- The raw dataset is large but sparse, so filtering inactive users and rarely rated books is necessary.
- Explicit ratings are concentrated toward positive values, which favors similarity-based taste matching.
- Popularity is useful as a homepage and cold-start fallback, but not sufficient as the main recommendation strategy.
- The user-book matrix is sparse and high-dimensional, making cosine similarity a practical and interpretable choice.
- The cleaned and filtered interaction data provides a stable foundation for the deployed recommendation app.